In [2]:
import pandas as pd
import numpy as np
import hashlib
import random
from datetime import timedelta

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── Load ──────────────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_csv('24hr_summary - Sheet1.csv')
print(f"  Original shape: {df.shape}")

# Drop the 1 null Cell row
df = df.dropna(subset=['Cell'])
print(f"  After dropping null Cell row: {df.shape}")

# ── Step 1: Anonymize Cell Names ──────────────────────────────
print("\nAnonymizing cell names...")

# Extract all unique region prefixes
cells = df['Cell'].unique()
prefixes = sorted(set(c.split('_')[0] for c in cells if '_' in c))
site_codes = sorted(set(c.split('_')[1] for c in cells if len(c.split('_')) >= 2))

print(f"  Real region prefixes found: {prefixes}")
print(f"  Total unique site codes: {len(site_codes)}")

# Map real region prefixes → generic region codes
# Use letters A, B, C... so no real geography is traceable
region_map = {}
for i, prefix in enumerate(prefixes):
    region_map[prefix] = f"REG_{chr(65 + i)}"  # REG_A, REG_B, REG_C ...

print(f"  Region mapping: {region_map}")

# Map real site codes → anonymous numeric codes
# Shuffle the site codes so order is not preserved
shuffled_indices = list(range(1, len(site_codes) + 1))
random.shuffle(shuffled_indices)
site_map = {code: f"S{str(idx).zfill(4)}" for code, idx in zip(site_codes, shuffled_indices)}

# Build full cell name mapping
def anonymize_cell(cell_name):
    parts = cell_name.split('_')
    if len(parts) == 3:
        region = region_map.get(parts[0], parts[0])
        site = site_map.get(parts[1], parts[1])
        sector = parts[2]  # sector number kept (1,2,3 not sensitive)
        return f"{region}_{site}_{sector}"
    elif len(parts) == 2:
        region = region_map.get(parts[0], parts[0])
        site = site_map.get(parts[1], parts[1])
        return f"{region}_{site}"
    else:
        # Fallback: hash it
        return "CELL_" + hashlib.md5(cell_name.encode()).hexdigest()[:6].upper()

df['Cell'] = df['Cell'].apply(anonymize_cell)
print(f"  Sample anonymized cells: {df['Cell'].unique()[:5].tolist()}")

# ── Step 2: Anonymize Vendor Names ───────────────────────────
print("\nAnonymizing vendor names...")

# Real vendor names → generic labels
# Order is shuffled so ZTE is not always Vendor_A
vendor_labels = ['Vendor_X', 'Vendor_Y', 'Vendor_Z']
real_vendors = sorted(df['Vendor'].unique().tolist())  # ['Huawei', 'Nokia', 'ZTE']
random.shuffle(vendor_labels)
vendor_map = {v: vendor_labels[i] for i, v in enumerate(real_vendors)}

print(f"  Vendor mapping applied (not disclosed in output): {vendor_map}")
df['Vendor'] = df['Vendor'].map(vendor_map)
print(f"  Anonymized vendor distribution:\n{df['Vendor'].value_counts().to_string()}")

# ── Step 4: Shift Dates ───────────────────────────────────────
print("\nShifting dates...")

df['Alarm Raised Date'] = pd.to_datetime(df['Alarm Raised Date'])

# Fixed random offset: shift all dates by 200–400 days into the past
# This hides when the real incidents happened
date_shift_days = random.randint(200, 400)
print(f"  Date shift applied: -{date_shift_days} days (hidden from output)")

df['Alarm Raised Date'] = df['Alarm Raised Date'] - timedelta(days=date_shift_days)
df['Alarm Raised Date'] = df['Alarm Raised Date'].dt.strftime('%Y-%m-%d')

print(f"  New date range: {df['Alarm Raised Date'].min()} to {df['Alarm Raised Date'].max()}")

# ── Step 3: Add Noise to Numeric Values ───────────────────────
print("\nAdding noise to numeric values...")

# Noise level: 3% gaussian noise — small enough to preserve distribution
# large enough to prevent reverse-engineering exact values
NOISE_LEVEL = 0.03

numeric_cols = ['Total_Inactive_Time_Seconds']
hourly_inactive = [f'Hour_{h}_Inactive' for h in range(1, 25)]
numeric_cols += hourly_inactive

for col in numeric_cols:
    original = df[col].values.astype(float)
    noise = np.random.normal(0, NOISE_LEVEL * np.maximum(original, 1), size=len(original))
    perturbed = np.round(original + noise).astype(int)
    df[col] = np.maximum(perturbed, 0)  # never go below 0

# Fluctuation counts: add ±1 noise with low probability (10% chance)
fluct_cols = [f'Hour_{h}_Fluctuation' for h in range(1, 25)] + ['Fluctuation_Count']
for col in fluct_cols:
    mask = np.random.random(len(df)) < 0.10  # 10% of rows get ±1
    delta = np.random.choice([-1, 1], size=len(df))
    df[col] = np.where(mask, np.maximum(df[col] + delta, 0), df[col])

print(f"  Noise added to {len(numeric_cols)} inactive columns and {len(fluct_cols)} fluctuation columns")

# ── Step 4: Recalculate Total_Inactive_Time_Seconds from hourly ─
# After noise, make Total consistent with sum of hourly values
print("\nRecalculating totals from perturbed hourly values...")
df['Total_Inactive_Time_Seconds'] = df[hourly_inactive].sum(axis=1)

# ── Step 5: Rebuild Inactive time string ──────────────────────
print("Rebuilding Inactive time strings...")

def seconds_to_string(s):
    s = int(s)
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h} Hours {m} Minutes {sec} Seconds"

df['Inactive time'] = df['Total_Inactive_Time_Seconds'].apply(seconds_to_string)

# ── Step 6: Recalculate Fluctuation_Count from hourly ─────────
df['Fluctuation_Count'] = df[[f'Hour_{h}_Fluctuation' for h in range(1, 25)]].sum(axis=1)

# ── Step 7: Slightly perturb Initial and Final Down Time ──────
print("Perturbing down times slightly...")

def perturb_time(time_str, max_seconds=30):
    """Shift time by up to 30 seconds randomly"""
    try:
        parts = time_str.split(':')
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
        total = h * 3600 + m * 60 + s
        delta = random.randint(-max_seconds, max_seconds)
        total = max(0, min(86399, total + delta))
        return f"{total//3600}:{str((total%3600)//60).zfill(2)}:{str(total%60).zfill(2)}"
    except:
        return time_str

df['Initial_Down_Time'] = df['Initial_Down_Time'].apply(perturb_time)
df['Final_Down_Time'] = df['Final_Down_Time'].apply(perturb_time)

# ── Step 8: Final Validation ──────────────────────────────────
print("\n=== VALIDATION ===")
print(f"Final shape: {df.shape}")
print(f"Null values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nSample anonymized data:")
print(df[['Cell', 'Alarm Raised Date', 'Vendor', 'Total_Inactive_Time_Seconds', 'Fluctuation_Count']].head(10).to_string())

print(f"\nVendor distribution:")
print(df['Vendor'].value_counts())

print(f"\nDate range after anonymization:")
print(f"  {df['Alarm Raised Date'].min()} to {df['Alarm Raised Date'].max()}")

print(f"\nTotal_Inactive_Time_Seconds stats:")
print(df['Total_Inactive_Time_Seconds'].describe().round(1))

# ── Save ──────────────────────────────────────────────────────
output_path = 'anonymized_alarm_data.csv'
df.to_csv(output_path, index=False)
print(f"\n✅ Anonymized dataset saved to '{output_path}'")
print(f"   Rows: {len(df)}, Columns: {len(df.columns)}")
print("\nIMPORTANT: The date_shift_days value is NOT saved anywhere.")
print("This means the original dates cannot be reverse-engineered.")


Loading dataset...
  Original shape: (14055, 56)
  After dropping null Cell row: (14054, 56)

Anonymizing cell names...
  Real region prefixes found: ['BAR', 'CTG', 'DHK', 'KHU', 'MYM', 'RAG', 'RAJ', 'SYL', 'TBTS']
  Total unique site codes: 3356
  Region mapping: {'BAR': 'REG_A', 'CTG': 'REG_B', 'DHK': 'REG_C', 'KHU': 'REG_D', 'MYM': 'REG_E', 'RAG': 'REG_F', 'RAJ': 'REG_G', 'SYL': 'REG_H', 'TBTS': 'REG_I'}
  Sample anonymized cells: ['REG_A_S2424_1', 'REG_A_S2424_2', 'REG_A_S2424_3', 'REG_A_S1581_1', 'REG_A_S1581_2']

Anonymizing vendor names...
  Vendor mapping applied (not disclosed in output): {'Huawei': 'Vendor_Z', 'Nokia': 'Vendor_Y', 'ZTE': 'Vendor_X'}
  Anonymized vendor distribution:
Vendor
Vendor_X    9281
Vendor_Z    4198
Vendor_Y     575

Shifting dates...
  Date shift applied: -343 days (hidden from output)
  New date range: 2024-06-08 to 2024-06-24

Adding noise to numeric values...
  Noise added to 25 inactive columns and 25 fluctuation columns

Recalculating totals from